# Ingredient Dataset — Merge & Clean (FV40 + FOOD-INGREDIENTS)
**Produces the standard merged dataset used by every downstream part of the project**
(YOLOv8, Faster R-CNN, and eventually the image → ingredients → recipe pipeline).

**v3 — no image copying.** v2 symlinked images into `merged_yolo/` to avoid duplicating the
dataset, but Google Drive's FUSE mount in Colab doesn't support `os.symlink` at all
(`OSError: [Errno 95] Operation not supported`), so it silently fell back to copying all 27,193
images one at a time — a multi-GB, many-minutes operation that looked like a hang.

This version keeps images at their **original locations** (FV40 / FOOD-INGREDIENTS folders) the
same way `instances.json` already does, and only ever writes small text files:

1. **YOLO label `.txt` files**, written next to each image in a sibling `labels/` folder that
   mirrors its original `images/` folder — the exact convention Ultralytics expects
   (`.../images/x.jpg` → `.../labels/x.txt`), so no directory restructuring is needed.
2. **`train.txt` / `val.txt` / `test.txt`** — plain lists of absolute image paths. Ultralytics
   accepts a path-list file in place of a directory for `train:`/`val:`/`test:` in `data.yaml`.

One consequence worth knowing about: FOOD-INGREDIENTS already has a `labels/` folder with the
*original* Roboflow class ids. Writing our unified-id labels there means overwriting those files,
so this notebook backs the originals up to `labels_original/` first (once — safe to re-run).

Still handles what `instances.json` doesn't:
1. **Split leakage** — 8.8% of FOOD-INGREDIENTS source photos have augmented copies scattered
   across train/valid/test in the raw `split` field.
2. **FV40 has no split at all** (`split: None` for every FV40 image) — created here, stratified
   80/10/10 by sub-collection.

**Output (all in `merged_yolo/` on Drive):** `data.yaml`, `train.txt`/`val.txt`/`test.txt`,
`split_resolution.csv`. The label `.txt` files themselves live next to the original images, not
in `merged_yolo/`. Downstream notebooks only ever need `data.yaml`.

Run this once (or whenever `instances.json` is updated, or the split logic changes) — it doesn't
need a GPU.

## 0. Setup

In [1]:
# Colab setup
from google.colab import drive
drive.mount('/content/drive')

!pip install pyyaml -q


Mounted at /content/drive


In [2]:
import json
import os
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import yaml

np.random.seed(42)

DATA_ROOT = Path("/content/drive/Shareddrives/Computer Vision Final Project/Data")
FOOD_DIR = DATA_ROOT / "food_ingredients"

# UPDATE THIS to wherever the teammate's merged annotations file actually lives on Drive
MERGED_ANNOTATIONS_PATH = DATA_ROOT / "merged" / "instances.json"

# Standard merged dataset artifacts (data.yaml, path-list files, manifest) live here.
# NOTE: this folder holds small text files only -- the images themselves stay in
# fv40_merged/ and food_ingredients/, they are never copied.
MERGED_DIR = DATA_ROOT / "merged_yolo"
MERGED_DIR.mkdir(parents=True, exist_ok=True)

assert MERGED_ANNOTATIONS_PATH.exists(), (
    f"{MERGED_ANNOTATIONS_PATH} not found -- update MERGED_ANNOTATIONS_PATH to match where "
    "the merged instances.json actually lives on Drive.")


## 1. Load the merged annotations + class taxonomy

Categories come straight from `instances.json` — no re-deriving. Ids 0-147 are contiguous, so the
`names` list for `data.yaml` is just the categories sorted by id.

Worth a quick manual skim of `categories` before training: a couple of names look like they might
need a second look (e.g. `cornflakec` is almost certainly a typo for `cornflakes`, and it's worth
double-checking `eggplant` vs `green brinjal` aren't meant to be the same ingredient). Flagging
these for the team rather than silently "fixing" them, since a wrong guess here corrupts labels.

In [3]:
with open(MERGED_ANNOTATIONS_PATH) as f:
    merged = json.load(f)

images_df = pd.DataFrame(merged["images"])
anns_df = pd.DataFrame(merged["annotations"])
categories_df = pd.DataFrame(merged["categories"]).sort_values("id")

assert list(categories_df["id"]) == list(range(len(categories_df))), \
    "Category ids aren't contiguous 0..N-1 -- data.yaml names list would be misaligned."

unified_classes = categories_df["name"].tolist()
print(f"{len(images_df):,} images | {len(anns_df):,} annotations | {len(unified_classes)} classes")
print(f"Source datasets: {images_df['source_dataset'].value_counts().to_dict()}")


27,193 images | 305,637 annotations | 148 classes
Source datasets: {'fv40': 17459, 'food_ingredients': 9734}


## 2. Split-leakage fix (FOOD-INGREDIENTS)

Augmented copies of the same source photo (`file_name.rf.<hash>` share a `source_id`) are still
scattered across train/valid/test in this file — confirmed 8.8% of source photos, matching the
original EDA. Reassign every augmented copy of a source photo to a **single split** — majority
split wins, ties go to train.

In [4]:
def food_source_id(file_path: str) -> str:
    return Path(file_path).name.split(".rf.")[0]

food_mask = images_df["source_dataset"] == "food_ingredients"
food_df = images_df.loc[food_mask].copy()
food_df["source_id"] = food_df["file_path"].apply(food_source_id)

def resolve_split(g):
    counts = g["split"].value_counts()
    if len(counts) == 1:
        return counts.index[0]
    top = counts[counts == counts.max()].index.tolist()
    return "train" if "train" in top else top[0]

resolved = food_df.groupby("source_id").apply(resolve_split, include_groups=False)
food_df["resolved_split"] = food_df["source_id"].map(resolved)

n_reassigned = (food_df["split"] != food_df["resolved_split"]).sum()
print(f"Reassigning {n_reassigned} images ({n_reassigned/len(food_df):.1%}) to fix split leakage")

# Roboflow's "valid" -> our "val"
food_df["final_split"] = food_df["resolved_split"].replace({"valid": "val"})
food_df[["id", "file_path", "split", "final_split", "source_id"]].head()


Reassigning 1025 images (10.5%) to fix split leakage


,id,file_path,split,final_split,source_id
17459,17459,/content/drive/Shareddrives/Computer Vision Fi...,train,train,-100_jpg
17460,17460,/content/drive/Shareddrives/Computer Vision Fi...,train,train,-100_jpg
17461,17461,/content/drive/Shareddrives/Computer Vision Fi...,train,train,-100_jpg
17462,17462,/content/drive/Shareddrives/Computer Vision Fi...,train,train,-100_jpg
17463,17463,/content/drive/Shareddrives/Computer Vision Fi...,train,train,-100_jpg


## 3. Create a split for FV40

Every FV40 image has `split: None` in the merged file. Stratified 80/10/10 by sub-collection
(parsed from the filename prefix, same grouping the EDA used) so each sub-collection is
represented in every split.

In [5]:
fv40_df = images_df.loc[images_df["source_dataset"] == "fv40"].copy()
fv40_df["subset"] = fv40_df["file_path"].apply(lambda p: Path(p).name.split("__")[0])

def stratified_split(df, group_col, ratios=(0.8, 0.1, 0.1), seed=42):
    rng = np.random.default_rng(seed)
    assignments = {}
    for _, group in df.groupby(group_col):
        ids = group["id"].tolist()
        rng.shuffle(ids)
        n = len(ids)
        n_train = int(n * ratios[0])
        n_val = int(n * ratios[1])
        for i in ids[:n_train]:
            assignments[i] = "train"
        for i in ids[n_train:n_train + n_val]:
            assignments[i] = "val"
        for i in ids[n_train + n_val:]:
            assignments[i] = "test"
    return assignments

fv40_split_map = stratified_split(fv40_df, "subset")
fv40_df["final_split"] = fv40_df["id"].map(fv40_split_map)
print(fv40_df["final_split"].value_counts())


final_split
train    13966
test      1749
val       1744
Name: count, dtype: int64


In [6]:
# Combine both sources back into one frame with a single resolved split column
resolved_images_df = pd.concat([
    fv40_df[["id", "file_path", "width", "height", "source_dataset", "final_split"]],
    food_df[["id", "file_path", "width", "height", "source_dataset", "final_split"]],
], ignore_index=True)

resolved_images_df.to_csv(MERGED_DIR / "split_resolution.csv", index=False)
print(resolved_images_df["final_split"].value_counts())
print(f"\nSaved -> {MERGED_DIR / 'split_resolution.csv'}")


final_split
train    23319
val       1972
test      1902
Name: count, dtype: int64

Saved -> /content/drive/Shareddrives/Computer Vision Final Project/Data/merged_yolo/split_resolution.csv


## 4. Back up FOOD-INGREDIENTS' original labels

FOOD-INGREDIENTS already has a `labels/` folder per split, using its own (non-unified) class ids.
We're about to overwrite those with unified-id labels, so rename the originals to
`labels_original/` first — once. Safe to re-run: skips any split that's already backed up.

In [7]:
for split in ["train", "valid", "test"]:
    labels_dir = FOOD_DIR / split / "labels"
    backup_dir = FOOD_DIR / split / "labels_original"
    if labels_dir.exists() and not backup_dir.exists():
        labels_dir.rename(backup_dir)
        print(f"Backed up {labels_dir} -> {backup_dir}")
    elif backup_dir.exists():
        print(f"{split}: already backed up, skipping")
    else:
        print(f"{split}: no labels/ folder found, skipping")


train: already backed up, skipping
valid: already backed up, skipping
test: already backed up, skipping


## 5. Write unified-id YOLO labels next to the original images

No copying — labels are written directly under each image's own dataset folder, in a `labels/`
sibling of its `images/` folder (Ultralytics' standard convention: `.../images/x.jpg` →
`.../labels/x.txt`, using the *last* `images` segment in the path, matching Ultralytics'
`img2label_paths` logic exactly so `data.yaml` doesn't need any extra config).

In [8]:
def img2label_path(img_path: Path) -> Path:
    """Mirrors ultralytics.data.utils.img2label_paths: swap the last /images/ for /labels/."""
    sa, sb = f"{os.sep}images{os.sep}", f"{os.sep}labels{os.sep}"
    img_str = str(img_path)
    return Path(sb.join(img_str.rsplit(sa, 1))).with_suffix(".txt")

def write_yolo_label(label_path: Path, boxes):
    """boxes: list of (class_id, x_center_norm, y_center_norm, w_norm, h_norm)"""
    label_path.parent.mkdir(parents=True, exist_ok=True)
    with open(label_path, "w") as f:
        for cid, xc, yc, w, h in boxes:
            f.write(f"{cid} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n")


In [9]:
# Group annotations per image for fast lookup -- plain tuples, not full namedtuples,
# to keep this fast even at 300K+ rows.
anns_by_image = defaultdict(list)
for image_id, cid, bbox in zip(anns_df["image_id"], anns_df["category_id"], anns_df["bbox"]):
    anns_by_image[image_id].append((cid, bbox))


In [10]:
img_meta = resolved_images_df.set_index("id")

import time
PROGRESS_EVERY = 500
start = time.time()

skipped = 0
written = 0
already_done = 0
valid_image_ids = []  # images that exist on disk (label written or already present)
for i, row in enumerate(img_meta.itertuples(), start=1):
    img_id = row.Index
    src_path = Path(row.file_path)
    label_path = img2label_path(src_path)

    # Resumable: skip images whose label was already written by a previous (interrupted) run.
    if label_path.exists():
        already_done += 1
        valid_image_ids.append(img_id)
    else:
        if not src_path.exists():
            skipped += 1
        else:
            valid_image_ids.append(img_id)
            boxes = []
            for cid, bbox in anns_by_image.get(img_id, []):
                x, y, w, h = bbox
                xc = (x + w / 2) / row.width
                yc = (y + h / 2) / row.height
                boxes.append((cid, xc, yc, w / row.width, h / row.height))
            write_yolo_label(label_path, boxes)
            written += 1

    if i % PROGRESS_EVERY == 0:
        elapsed = time.time() - start
        rate = i / elapsed
        remaining = (len(img_meta) - i) / rate
        print(f"{i}/{len(img_meta)}  written={written}  already_done={already_done}  "
              f"skipped={skipped}  ({rate:.1f} img/s, ~{remaining/60:.1f} min left)")

print(f"Done. Wrote {written} new labels, {already_done} already existed, "
      f"skipped {skipped} missing files (check Drive sync if this is nonzero).")


500/27193  written=0  already_done=498  skipped=2  (5.5 img/s, ~80.2 min left)
1000/27193  written=0  already_done=998  skipped=2  (11.1 img/s, ~39.4 min left)
1500/27193  written=0  already_done=1498  skipped=2  (16.6 img/s, ~25.8 min left)
2000/27193  written=0  already_done=1998  skipped=2  (22.1 img/s, ~19.0 min left)
2500/27193  written=0  already_done=2498  skipped=2  (27.6 img/s, ~14.9 min left)
3000/27193  written=0  already_done=2998  skipped=2  (33.0 img/s, ~12.2 min left)
3500/27193  written=0  already_done=3498  skipped=2  (38.5 img/s, ~10.3 min left)
4000/27193  written=0  already_done=3998  skipped=2  (43.9 img/s, ~8.8 min left)
4500/27193  written=0  already_done=4498  skipped=2  (49.3 img/s, ~7.7 min left)
5000/27193  written=0  already_done=4998  skipped=2  (54.7 img/s, ~6.8 min left)
5500/27193  written=0  already_done=5498  skipped=2  (60.1 img/s, ~6.0 min left)
6000/27193  written=0  already_done=5998  skipped=2  (65.4 img/s, ~5.4 min left)
6500/27193  written=0  al

## 6. Build `train.txt` / `val.txt` / `test.txt` path lists

Ultralytics accepts a text file of absolute image paths in place of a directory for
`train:`/`val:`/`test:`. This is what actually replaces the symlink/copy step — no image ever
moves, we just tell Ultralytics where to find each split.

In [11]:
for split in ["train", "val", "test"]:
    paths = img_meta.loc[(img_meta["final_split"] == split) & (img_meta.index.isin(valid_image_ids)), "file_path"]
    list_path = MERGED_DIR / f"{split}.txt"
    list_path.write_text("\n".join(paths) + "\n")
    print(f"{split:5s}  {len(paths):6d} images -> {list_path}")


train   23317 images -> /content/drive/Shareddrives/Computer Vision Final Project/Data/merged_yolo/train.txt
val      1972 images -> /content/drive/Shareddrives/Computer Vision Final Project/Data/merged_yolo/val.txt
test     1902 images -> /content/drive/Shareddrives/Computer Vision Final Project/Data/merged_yolo/test.txt


## 7. Sanity check: labels exist for a sample of each split

Since labels were written next to the images rather than into `merged_yolo/`, spot-check a
handful per split to confirm `img2label_path` resolved correctly before training on this.

In [12]:
import random
random.seed(0)

for split in ["train", "val", "test"]:
    paths = img_meta.loc[(img_meta["final_split"] == split) & (img_meta.index.isin(valid_image_ids)), "file_path"].tolist()
    sample = random.sample(paths, min(5, len(paths)))
    n_found = sum(img2label_path(Path(p)).exists() for p in sample)
    print(f"{split:5s}  {n_found}/{len(sample)} sampled labels found")


train  5/5 sampled labels found
val    5/5 sampled labels found
test   5/5 sampled labels found


## 8. Write the standard `data.yaml`

This is the file every downstream notebook (YOLO, Faster R-CNN, etc.) should load — nobody else needs to touch `instances.json`, FV40, or FOOD-INGREDIENTS raw files again.

In [13]:
data_yaml = {
    "train": str(MERGED_DIR / "train.txt"),
    "val": str(MERGED_DIR / "val.txt"),
    "test": str(MERGED_DIR / "test.txt"),
    "nc": len(unified_classes),
    "names": unified_classes,
}

data_yaml_path = MERGED_DIR / "data.yaml"
with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print(f"Wrote {data_yaml_path} -- {len(unified_classes)} classes, ready for training notebooks")


Wrote /content/drive/Shareddrives/Computer Vision Final Project/Data/merged_yolo/data.yaml -- 148 classes, ready for training notebooks


## Done — standard dataset artifacts

Everything downstream (YOLOv8, Faster R-CNN, recipe-matching pipeline) should point at:

| Artifact | Path |
|---|---|
| Ultralytics-ready config | `merged_yolo/data.yaml` |
| Split path lists | `merged_yolo/{train,val,test}.txt` |
| Split-leakage fix + FV40 split record | `merged_yolo/split_resolution.csv` |
| Labels (unified ids) | next to each image, in that image's own `labels/` folder |
| FOOD-INGREDIENTS' original labels (backup) | `food_ingredients/{split}/labels_original/` |

**Images were never copied or moved** — `merged_yolo/` only holds small text files. FV40 and
FOOD-INGREDIENTS stay exactly where they already live on Drive.

Labels are standard YOLO txt format (`class_id x_center y_center width height`, all normalized
0-1), so **Faster R-CNN can also read this** — convert to absolute `(x1, y1, x2, y2)` at load time.
`data.yaml`'s `names` list gives the class-id ↔ name mapping (same ids as `instances.json`'s
`categories`, so anything already built against that file stays consistent).

Re-run this notebook only if `instances.json` changes or the split logic needs revisiting — it's
safe to re-run (backups and label writes are idempotent / overwrite cleanly).

**Flag for the team:** `cornflakec` in the class list is almost certainly a typo for `cornflakes` —
worth a quick fix in the source before the report screenshots the class list.